# 02: Probability, from counting to Bayes

**Practice set:** `practice/02-probability.md` · **Exam tasks:** Jan 18, "The Pit-Stop Paradox" (page 9) and "The Blind Board Gambit" (page 7) · **Moves:** M2, M16, M18

## Hook

Your delivery robot has a cheap IR obstacle sensor on its front bumper. The datasheet says it fires on **90%** of real obstacles. It also false-fires on **10%** of clear readings, because sunlight on a shiny floor fools it. On your test route, an obstacle is really ahead in about **1 reading in 20**.

The sensor beeps. Should the robot slam on the brakes?

If it brakes on every beep, it stutters down the hallway. If it ignores beeps, it hits things. To decide, you need one number: *given that it beeped, how likely is it that an obstacle is really there?*

**PREDICT now.** Write a number on paper: when the sensor beeps, what is the chance there really is an obstacle? Keep it. Section 4 settles it.

The robot also has an ultrasonic sensor. If *both* fire, how sure should it be? And how would you combine dozens of weak clues at once? This notebook builds all of it from one habit: **imagine a crowd of cases and count.**

## Before you start

- **You need:** fractions and percentages. Section 3 of notebook 01 (the gripper mystery) helps, but you don't need it.
- **How to use this notebook:** run the cells in order. When you see **PREDICT**, write your guess *before* running the next cell. When you see **BUILD**, write the function yourself; the test cell tells you when it's right. Hints sit in collapsed blocks. Open one only after a real attempt.
- **Pen and paper stay on the desk.** The exam has no computer. Use the notebook to check and explore *after* you've reasoned it out.

In [ ]:
import prep
import math
import itertools
from collections import Counter
import numpy as np
import matplotlib.pyplot as plt

---
## 1. Probability is counting

**Idea.** A probability is a fraction of cases. When every outcome is equally likely, you just count:

**P(event) = (number of outcomes where the event happens) ÷ (number of possible outcomes)**

Four words you'll use all the time:
- **outcome**: one complete result. Rolling two dice and getting (2, 5) is one outcome.
- **event**: a yes/no question about the outcome, such as "is the sum 7?".
- **P(A)**: the probability of event A. It runs from 0 (never happens) to 1 (always happens).
- **not A**: the event "A did not happen". Every outcome lands in exactly one of A and not A, so **P(not A) = 1 − P(A)**.

**Smallest example.** Roll two dice. There are 6 × 6 = 36 equally likely outcomes. The grid shows the sum for each one.

In [ ]:
outcomes = list(itertools.product(range(1, 7), repeat=2))   # (first die, second die)
grid = [[a] + [a + b for b in range(1, 7)] for a in range(1, 7)]
prep.table(grid, header=["first die \\ second die", 1, 2, 3, 4, 5, 6])
print(len(outcomes), "outcomes")

**PREDICT:**
1. Which sum appears most often in the grid? What is its probability?
2. What is P(at least one six)? Count it in the grid. Then get the same number a second way, using "not".

<details><summary>Hint for question 2</summary>

"At least one six" is the opposite of "no six at all". Each die has 5 faces that aren't a six. How many outcomes have no six?

</details>

In [ ]:
sums = Counter(a + b for a, b in outcomes)
prep.table([[s, sums[s], f"{sums[s]}/36"] for s in sorted(sums)], header=["sum", "outcomes", "probability"])
at_least_one_six = sum(1 for a, b in outcomes if a == 6 or b == 6)
no_six = sum(1 for a, b in outcomes if a != 6 and b != 6)
print(f"\nat least one six: {at_least_one_six}/36")
print(f"no six at all:    {no_six}/36  (5 choices for each die: 5 x 5)")
print(f"1 - {no_six}/36 = {36 - no_six}/36")

Two habits to keep:
- **"At least one" questions are usually easiest as 1 − P(none).** You'll need this in the Pit-Stop walkthrough.
- **When outcomes are *not* equally likely, imagine a crowd.** "P(rain) = 0.3" means: picture 1,000 identical days; about 300 are rainy. Split an imagined 1,000 cases in the right proportions and every later question turns back into counting. Every section of this notebook uses this trick.

### BUILD 1: `prob`
`records` is a list of equally likely cases (dice outcomes, or rows of a robot log). `event` is a function that takes one record and returns True or False. Return the fraction of records where `event` is True.

In [ ]:
def prob(records, event):
    # your code here
    raise NotImplementedError

In [ ]:
prep.test(prob, [
    ((outcomes, lambda o: o[0] + o[1] == 7), 6/36),
    ((outcomes, lambda o: 6 in o), 11/36),
    (([1, 2, 3, 4], lambda x: x > 1), 0.75),
])

---
## 2. Conditional probability: zoom into a group

**Idea.** **P(A | B)** is read "the probability of A given B". It means: **throw away every case where B didn't happen, then ask what fraction of the cases left over have A.** The bar "|" works like a zoom lens. It shrinks the denominator down to the group B.

This is the "which group?" question from notebook 01. Recall divided by the truly-YES group; precision divided by the said-YES group. Same counts, different zoom.

**Smallest example.** Your robot logged 12 runs. For each one you noted the floor and whether a wheel slipped.

In [ ]:
runs = [
    {"floor": "tile",   "slip": True},  {"floor": "carpet", "slip": False}, {"floor": "tile",   "slip": False},
    {"floor": "tile",   "slip": True},  {"floor": "carpet", "slip": True},  {"floor": "tile",   "slip": False},
    {"floor": "tile",   "slip": True},  {"floor": "carpet", "slip": False}, {"floor": "tile",   "slip": False},
    {"floor": "tile",   "slip": True},  {"floor": "carpet", "slip": False}, {"floor": "tile",   "slip": False},
]
prep.table([[i, r["floor"], r["slip"]] for i, r in enumerate(runs, start=1)], header=["run", "floor", "slip"])

**PREDICT:** work out both numbers from the table before running the next cell.
- P(slip | tile): of the tile runs, what fraction slipped?
- P(tile | slip): of the runs that slipped, what fraction were on tile?

Are they the same number? Why or why not?

In [ ]:
tile_runs = [r for r in runs if r["floor"] == "tile"]
slip_runs = [r for r in runs if r["slip"]]
print("Zoom into TILE runs:", len(tile_runs), "of them")
print("   of these, slipped:", sum(r["slip"] for r in tile_runs))
print(f"   P(slip | tile) = {sum(r['slip'] for r in tile_runs)}/{len(tile_runs)}")
print()
print("Zoom into SLIP runs:", len(slip_runs), "of them")
print("   of these, on tile:", sum(r["floor"] == "tile" for r in slip_runs))
print(f"   P(tile | slip) = {sum(r['floor'] == 'tile' for r in slip_runs)}/{len(slip_runs)}")

**The aha: a conditional probability is a count divided by the size of the group you zoomed into.** P(slip | tile) and P(tile | slip) share the same top count (4 runs that are tile *and* slip), but they divide by different groups (8 tile runs vs 5 slip runs). So they're different numbers. Mixing them up is the most common probability mistake there is, and the hook is built on it.

**The multiplication rule** (you'll use it constantly). Count the runs that are tile **and** slip: 4 of 12. Now get there in two steps. First zoom into tile (8 of 12 runs), then take the fraction of those that slipped (4 of 8):

(8/12) × (4/8) = 4/12

The 8s cancel. In symbols: **P(A and B) = P(B) × P(A | B)**. It isn't a new law, just the zoom done in two steps.

### BUILD 2: `cond_prob`
Return P(event | given): the fraction of the records satisfying `given` that also satisfy `event`.

In [ ]:
def cond_prob(records, event, given):
    # your code here
    raise NotImplementedError

In [ ]:
prep.test(cond_prob, [
    ((runs, lambda r: r["slip"], lambda r: r["floor"] == "tile"), 0.5),
    ((runs, lambda r: r["floor"] == "tile", lambda r: r["slip"]), 0.8),
    ((outcomes, lambda o: o[0] + o[1] == 12, lambda o: o[0] == 6), 1/6),
    ((outcomes, lambda o: o[0] == 6, lambda o: o[0] + o[1] == 12), 1.0),
])

---
## 3. Independence: test it with a table

**Idea.** A and B are **independent** if learning that B happened doesn't change the chance of A: P(A | B) = P(A). Multiply both sides by P(B) and use the multiplication rule to get the version that's easiest to check:

**A and B are independent exactly when P(A and B) = P(A) × P(B).**

To test it, put the counts in a 2×2 table: rows are A / not A, columns are B / not B. Independence means every row splits between the columns in the same proportions.

**PREDICT:** roll two dice again.
1. The sum obviously depends on the first die. So is "sum is 7" independent of "first die is even"?
2. Is "sum ≥ 10" independent of "first die is 6"?

Write yes or no for each, then run the cell.

In [ ]:
def table_2x2(A, B):
    t = [[0, 0], [0, 0]]
    for o in outcomes:
        t[0 if A(o) else 1][0 if B(o) else 1] += 1
    return t

for name, A, B in [("A = first die even,  B = sum is 7", lambda o: o[0] % 2 == 0, lambda o: o[0] + o[1] == 7),
                   ("A = first die is 6,  B = sum >= 10", lambda o: o[0] == 6, lambda o: o[0] + o[1] >= 10)]:
    t = table_2x2(A, B)
    print(name)
    prep.table([["A", t[0][0], t[0][1], t[0][0] / sum(t[0])],
                ["not A", t[1][0], t[1][1], t[1][0] / sum(t[1])]],
               header=["", "B", "not B", "fraction of row in B"])
    p_a, p_b = sum(t[0]) / 36, (t[0][0] + t[1][0]) / 36
    print(f"P(A and B) = {t[0][0]}/36 = {t[0][0] / 36:.4f}    P(A) x P(B) = {p_a:.4f} x {p_b:.4f} = {p_a * p_b:.4f}\n")

**Independence is a fact about the numbers, not about the story.** The first die *does* feed into the sum. But whatever the first die shows, exactly one face of the second die makes 7. So knowing the first die tells you nothing about "sum is 7". Never assume independence from the story; check it with the table.

**A weaker kind you'll need in sections 5 and 6: conditional independence.** Two sensors can be dependent overall, because both tend to fire when there's an obstacle. Yet they can be independent *once you know the truth*: given that there is an obstacle, whether the IR fired tells you nothing extra about whether the ultrasonic fires. Tiny example: two thermometers both read high on hot days, so their readings are correlated. On any one day, though, their small random errors have nothing to do with each other.

### BUILD 3: `is_independent`
`t` is a 2×2 table of counts: `[[n(A and B), n(A, not B)], [n(not A, B), n(not A, not B)]]`. Return True if P(A and B) = P(A) × P(B). Use `math.isclose` so rounding errors don't fool you.

<details><summary>Hint</summary>

P(A) is the first row's total ÷ everything. P(B) is the first column's total ÷ everything. For a 2×2 table, checking the one top-left cell is enough. (Can you see why? If the top-left cell matches, the other three cells are forced by the row and column totals.)

</details>

In [ ]:
def is_independent(t):
    # your code here
    raise NotImplementedError

In [ ]:
prep.test(is_independent, [
    (([[10, 30], [15, 45]],), True),
    (([[10, 30], [30, 30]],), False),
    (([[3, 15], [3, 15]],), True),
    (([[3, 3], [3, 27]],), False),
])

Keep this test in mind for option D of practice problem 2.5, and for a surprise in the Pit-Stop walkthrough.

---
## 4. The joint table, total probability and Bayes' rule

**Idea.** The truth is hidden (H = "an obstacle is ahead"). You see evidence (E = "the IR sensor beeps"). The datasheet gives you P(E | H) and P(E | not H). Those numbers zoom into the *truth* groups. But the robot needs P(H | E), which zooms into the *evidence* group. Section 2 warned you these are different numbers. To get from one to the other, build the **joint table**: every combination of (truth, evidence), as counts in 1,000 imagined readings.

Three names:
- **prior** P(H): what you believe *before* the evidence. Here 1 reading in 20, so 0.05.
- **likelihoods** P(E | H) and P(E | not H): how often the evidence shows up when H is true (0.90), and when it is false (0.10, the false-alarm rate).
- **posterior** P(H | E): what you believe *after* seeing the evidence. This is the number the robot needs.

**PREDICT:** on paper, fill in this table for 1,000 readings. Then compute P(obstacle | beep). How does it compare with your guess from the hook?

| truth | beep | silent | row total |
|---|---|---|---|
| obstacle | ? | ? | ? |
| clear | ? | ? | ? |

<details><summary>Hint 1</summary>

Fill the row totals first: 5% of 1,000 readings have an obstacle.

</details>

<details><summary>Hint 2</summary>

The sensor beeps on 90% of the obstacle row and on 10% of the clear row. The question zooms into the **beep column**.

</details>

In [ ]:
N = 1000
prior, p_beep_obstacle, p_beep_clear = 0.05, 0.90, 0.10
obstacle, clear = prior * N, (1 - prior) * N
cells = [["obstacle", obstacle * p_beep_obstacle, obstacle * (1 - p_beep_obstacle), obstacle],
         ["clear", clear * p_beep_clear, clear * (1 - p_beep_clear), clear]]
prep.table(cells, header=["truth", "beep", "silent", "row total"])
beeps = cells[0][1] + cells[1][1]
print(f"\nbeep column total = {cells[0][1]:.0f} + {cells[1][1]:.0f} = {beeps:.0f}")
print(f"P(obstacle | beep) = {cells[0][1]:.0f} / {beeps:.0f} = {cells[0][1] / beeps:.3f}")

Roughly **1 beep in 3** is a real obstacle, even though the sensor is "90% accurate". The obstacle row is small (50 readings), so 10% of the big clear row (95 false alarms) outnumbers 90% of the small row (45 real hits). How did your hook guess do?

**Now turn the counting into a formula.** Do each step on paper before you open the check.
1. Write the top-left cell (obstacle *and* beep) using N, P(H) and P(E | H).
2. Write the beep column total.
3. Write P(H | E) as a ratio. What happens to N?

<details><summary>Check your three formulas</summary>

1. N · P(H) · P(E | H). This is the multiplication rule from section 2: the size of the row times the fraction of it that beeps.
2. N · P(H) · P(E | H) + N · P(not H) · P(E | not H). Adding up every way the evidence can happen is called the **law of total probability**.
3. P(H | E) = P(H) · P(E | H) ÷ [ P(H) · P(E | H) + P(not H) · P(E | not H) ]. The N cancels.

That's **Bayes' rule**. You didn't memorise it; you read it off a table.

</details>

**The aha: the posterior is one cell of the joint table divided by its column total.** Whenever you forget the formula, draw the table.

### BUILD 4: `posterior`
Return P(H | E), given P(H), P(E | H) and P(E | not H).

In [ ]:
def posterior(prior, p_e_given_h, p_e_given_not_h):
    # your code here
    raise NotImplementedError

In [ ]:
prep.test(posterior, [
    ((0.05, 0.90, 0.10), 45/140),
    ((0.5, 0.8, 0.2), 0.8),
    ((0.3, 0.5, 0.5), 0.3),
    ((0.2, 1.0, 0.0), 1.0),
])

Look at the third test. The evidence is equally likely whether H is true or not, so the posterior equals the prior. Evidence only moves you if it's more likely under one side than the other.

### EXPLORE: the same beep in different places
The base rate (prior) depends on *where* the robot is: an empty hallway has few obstacles, a cluttered workshop has many.

**PREDICT:** at what prior does a beep first make "obstacle" more likely than not (posterior above 0.5)? Write a number, then run.

In [ ]:
if prep.ready(posterior, 0.5, 0.5, 0.5):
    priors_ = np.linspace(0.001, 0.5, 200)
    post_ = [posterior(p, 0.90, 0.10) for p in priors_]
    plt.figure(figsize=(6, 3.5))
    plt.plot(priors_, post_, label="P(obstacle | beep)")
    plt.plot(priors_, priors_, "--", color="gray", label="prior (no evidence)")
    plt.axhline(0.5, color="k", lw=0.5)
    plt.xlabel("prior P(obstacle)"); plt.ylabel("probability")
    plt.title("Same sensor, different rooms"); plt.legend(); plt.grid(alpha=.3)
    plt.show()

The crossing is where the two beep cells are equal: 0.9 · p = 0.1 · (1 − p). The sensor didn't change. Only the base rate did.

### EXPLORE: Monte Carlo vs exact Bayes
**Monte Carlo simulation** means estimating a probability by simulating many random cases on the computer and counting. It's the computer's version of "imagine 1,000 cases". It needs no formula, only the story, so it's a good way to check a Bayes answer when you aren't sure the formula fits.

**PREDICT:** if you simulate only 100 readings, about how many beeps will there be? How far from the exact answer could the estimate land?

In [ ]:
if prep.ready(posterior, 0.5, 0.5, 0.5):
    rng = np.random.default_rng(0)
    exact = posterior(0.05, 0.90, 0.10)
    rows = []
    for n in [100, 1_000, 10_000, 100_000, 1_000_000]:
        has_obstacle = rng.random(n) < 0.05
        beep = np.where(has_obstacle, rng.random(n) < 0.90, rng.random(n) < 0.10)
        est = (has_obstacle & beep).sum() / beep.sum()
        rows.append([n, int(beep.sum()), int((has_obstacle & beep).sum()), est, est - exact])
    prep.table(rows, header=["readings", "beeps", "beeps with obstacle", "estimate", "estimate - exact"])
    print(f"exact (Bayes): {exact:.4f}")

With 100 readings there are only about 14 beeps, so one obstacle more or less moves the estimate by about 0.07. The error shrinks as the crowd grows, but slowly: to make it about 10 times smaller you need about 100 times more readings. The exam gives you no computer, which is why the exact table matters.

### More than two hidden possibilities
Often the hidden truth has more than two options. Same table, more rows.

Your robot drives on tile 50% of the time, carpet 30% and a ramp 20%. Its wheels slip with probability 0.02 on tile, 0.10 on carpet and 0.25 on the ramp. **It just slipped. Which surface is it on?**

**PREDICT:** the ramp is the *least* common surface. Is it the most likely explanation for the slip?

In [ ]:
surfaces = {"tile": (0.50, 0.02), "carpet": (0.30, 0.10), "ramp": (0.20, 0.25)}
rows, total = [], 0
for s, (pr, lik) in surfaces.items():
    rows.append([s, pr, lik, pr * lik])
    total += pr * lik
for r in rows:
    r.append(r[3] / total)
prep.table(rows, header=["surface", "prior", "P(slip | surface)", "prior x likelihood", "÷ total = posterior"])
print(f"total = P(slip) = {total:.3f}")

The recipe never changes: **list every hidden scenario, multiply along each branch (prior × likelihood), keep the column that matches what you observed, divide by its total.** The ramp wins because its likelihood is big enough to beat its small prior.

### BUILD 5: `posteriors`
`priors` and `likelihoods` are dicts with the same keys (the scenarios). Return a dict mapping each scenario to its posterior.

In [ ]:
def posteriors(priors, likelihoods):
    # your code here
    raise NotImplementedError

In [ ]:
prep.test(posteriors, [
    (({"tile": 0.5, "carpet": 0.3, "ramp": 0.2}, {"tile": 0.02, "carpet": 0.10, "ramp": 0.25}),
     {"tile": 1/9, "carpet": 1/3, "ramp": 5/9}),
    (({"a": 0.5, "b": 0.5}, {"a": 0.2, "b": 0.6}), {"a": 0.25, "b": 0.75}),
    (({"h": 0.05, "not h": 0.95}, {"h": 0.9, "not h": 0.1}), {"h": 45/140, "not h": 95/140}),
])

**Now do practice problems 2.1(a) and 2.2 on paper**, then check them.

<details><summary>Hint 1 for 2.2</summary>

What are the hidden scenarios? "Lift or stairs" isn't enough, because P(late | stairs) depends on whether the lift was working.

</details>

<details><summary>Hint 2 for 2.2</summary>

There are three scenarios: (lift works, takes L), (lift works, takes S), (lift broken, takes S). Each one's prior is a product along its branch. Then follow the recipe above.

</details>

In [ ]:
prep.check("2.1a", ...)   # floor(100 x P(AI-written | flagged))
prep.check("2.2", ...)    # floor(100 x P(lift broken | late))

---
## 5. Updating twice

**Idea.** After the IR beep, the robot believes P(obstacle) ≈ 0.32. Now a second, *different* sensor, the ultrasonic, also says "obstacle". It fires on **80%** of real obstacles and false-fires on **5%** of clear readings. What should the robot believe now?

This only works cleanly if the two sensors are **conditionally independent given the truth** (section 3). That's reasonable for IR (light) and ultrasonic (sound): different physics, different false alarms. It is *not* reasonable for reading the same IR sensor twice, since the same sunlight would fool it both times.

**PREDICT:**
1. Your guess for P(obstacle | both fire).
2. Where should the second update start: from 0.05 or from 0.32?
3. Does the order matter (ultrasonic first, then IR)?

In [ ]:
N = 1000
both_obstacle = N * 0.05 * 0.90 * 0.80     # obstacle row, fraction where BOTH fire
both_clear    = N * 0.95 * 0.10 * 0.05     # clear row, fraction where BOTH fire
print("One big table, column 'both fire':")
prep.table([["obstacle", N * 0.05, "x 0.90 x 0.80", both_obstacle],
            ["clear",    N * 0.95, "x 0.10 x 0.05", both_clear]],
           header=["truth", "row total", "both fire", "count"])
print(f"P(obstacle | both) = {both_obstacle} / {both_obstacle + both_clear} = {both_obstacle / (both_obstacle + both_clear):.4f}\n")

if prep.ready(posterior, 0.5, 0.5, 0.5):
    step1 = posterior(0.05, 0.90, 0.10)
    step2 = posterior(step1, 0.80, 0.05)
    print(f"Two updates in a row: 0.05 -> IR -> {step1:.4f} -> ultrasonic -> {step2:.4f}")
    rev1 = posterior(0.05, 0.80, 0.05)
    rev2 = posterior(rev1, 0.90, 0.10)
    print(f"Other order:          0.05 -> ultrasonic -> {rev1:.4f} -> IR -> {rev2:.4f}")

**Today's posterior is tomorrow's prior.** When the pieces of evidence are conditionally independent, updating one at a time gives the same answer as one big table, in any order.

Why? In the big table, the "both fire" count for obstacles is N · P(H) · P(E₁ | H) · P(E₂ | H). Group the first factors: [N · P(H) · P(E₁ | H)] · P(E₂ | H). The bracket is exactly the beep column of the first update. The second update just splits that column again.

### BUILD 6: `update_many`
`evidence` is a list of `(p_e_given_h, p_e_given_not_h)` pairs, one per independent piece of evidence. Return the list of beliefs: the prior, then the belief after each update.

In [ ]:
def update_many(prior, evidence):
    # your code here
    raise NotImplementedError

In [ ]:
prep.test(update_many, [
    ((0.5, [(0.8, 0.2)]), [0.5, 0.8]),
    ((0.5, [(0.8, 0.2), (0.8, 0.2)]), [0.5, 0.8, 16/17]),
    ((0.2, []), [0.2]),
    ((0.05, [(0.9, 0.1), (0.8, 0.05)]), [0.05, 45/140, 36/40.75]),
])

### EXPLORE: odds, and many weak clues
The **odds** of an event are P ÷ (1 − P). For example, P = 0.8 gives odds 4 ("4 to 1"). In odds, each update is a single multiplication: **posterior odds = prior odds × P(E | H) ÷ P(E | not H)**. The ratio P(E | H) ÷ P(E | not H) is called the **likelihood ratio**. (Check it on the table above: divide the two cells of the beep column.)

A weak sensor fires on 60% of obstacles and 40% of clear readings, a likelihood ratio of only 1.5.

**PREDICT:** starting from 0.05, how many independent beeps from weak sensors like this does it take before the robot believes P(obstacle) > 0.5?

In [ ]:
if prep.ready(update_many, 0.5, []):
    beliefs = update_many(0.05, [(0.6, 0.4)] * 12)
    prep.table([[i, b, b / (1 - b)] for i, b in enumerate(beliefs)], header=["beeps", "P(obstacle)", "odds"])
    plt.figure(figsize=(6, 3.5))
    plt.plot(range(len(beliefs)), beliefs, marker="o")
    plt.axhline(0.5, color="gray", ls="--")
    plt.xlabel("number of independent weak beeps"); plt.ylabel("P(obstacle)")
    plt.title("Weak clues add up"); plt.grid(alpha=.3)
    plt.show()

Each beep multiplies the odds by 1.5. Starting from odds 1/19, you need enough factors of 1.5 to get past 1. Hold on to "many weak clues multiply up": it's the whole idea behind the next section.

**Now do practice problem 2.1(b) on paper.** Try both routes (one big table, and two updates in a row). They must agree.

In [ ]:
prep.check("2.1b", ...)   # floor(100 x P(AI-written | both flag))

---
## 6. Naive Bayes: many clues at once

**Idea.** Your robot writes log messages, and you want to flag the ones that need attention: **alert** vs **normal**. Each message is described by which words from a small vocabulary it contains. Every word is a clue, so you update once per word, exactly as in section 5. That requires the words to be conditionally independent given the class. That's almost never really true (words come in phrases), which is why the method is called **naive**. It still works surprisingly well.

**Bernoulli Naive Bayes**, written as a recipe. For each class c:
1. Start with the prior P(c).
2. For **every** word w in the vocabulary, multiply by P(w | c) if the word is present, or by **1 − P(w | c)** if it is absent.
3. You now have one score per class. Divide each score by the sum of the scores, so the results add to 1. (That's the "divide by the column total" step from section 4. People write it as P(c | message) **∝** P(c) · P(message | c), where ∝ means "proportional to": equal after you rescale so everything adds to 1.)

**Training data.** 4 alert logs and 8 normal logs. The table counts how many logs in each class contain each word.

In [ ]:
class_sizes = {"alert": 4, "normal": 8}
word_counts = {"alert":  {"error": 3, "battery": 2, "ok": 0},
               "normal": {"error": 1, "battery": 2, "ok": 6}}
prep.table([[w, f"{word_counts['alert'][w]}/4", f"{word_counts['normal'][w]}/8"] for w in word_counts["alert"]],
           header=["word", "fraction of alert logs", "fraction of normal logs"])

A new log contains **"error"**, and does **not** contain "battery" or "ok".

**PREDICT:** "ok" is absent from this log. Does an *absent* word tell you anything? Which class does its absence favour, and by roughly what factor?

In [ ]:
new_log = {"error"}
score = {}
rows = []
for c in class_sizes:
    s = class_sizes[c] / sum(class_sizes.values())
    rows.append([c, "prior", "", f"{class_sizes[c]}/12", s])
    for w, k in word_counts[c].items():
        p = k / class_sizes[c]
        factor = p if w in new_log else 1 - p
        s *= factor
        rows.append([c, w, "present" if w in new_log else "absent", f"{'p' if w in new_log else '1 - p'} = {factor:.4g}", s])
    score[c] = s
prep.table(rows, header=["class", "step", "word is", "factor", "running score"])
z = sum(score.values())
print()
for c in score:
    print(f"P({c} | log) = {score[c]:.6f} / {z:.6f} = {score[c] / z:.4f}")

**Absent words are evidence too.** "ok" appears in 6 of 8 normal logs, so a log *without* it is unusual for a normal log (factor 2/8) but ordinary for an alert log (factor 1). Forgetting the (1 − p) factor for absent words is the most common Naive Bayes mistake.

### The zero problem
A second log contains **"error" and "ok"** (not "battery").

**PREDICT:** no alert log in the training data contained "ok". What does the recipe say P(alert | this log) is? Is that sensible?

In [ ]:
log2 = {"error", "ok"}
for c in class_sizes:
    factors = [(w, (k / class_sizes[c]) if w in log2 else 1 - k / class_sizes[c]) for w, k in word_counts[c].items()]
    print(c, " prior", f"{class_sizes[c]}/12", " factors:", ", ".join(f"{w} {f:.3g}" for w, f in factors))

The alert score has a factor of exactly 0 in it, so P(alert) = 0, no matter how alarming the other words are. **One word that never appeared in training vetoes everything.** With only 4 alert logs, "never seen" really just means "not seen *yet*".

**Laplace smoothing** fixes this. Pretend each class saw one extra message *with* the word and one extra *without* it:

**P(w | c) = (count + 1) ÷ (N_c + 2)**, where N_c is the number of training messages in class c.

The "+2" is there because a word has two outcomes, present or absent. Tiny example: 0/4 becomes 1/6, and 4/4 becomes 5/6. Nothing is ever exactly 0 or 1 any more. Smoothing is applied to the word probabilities only, not to the class priors.

### BUILD 7: `naive_bayes`
- `class_sizes`: number of training messages per class (the prior is size ÷ total).
- `word_counts[c][w]`: how many training messages of class c contain word w. The vocabulary is the set of keys.
- `present`: the set of words in the new message. Every other vocabulary word is absent.
- `smoothing`: if True, use (count + 1) ÷ (N_c + 2).

Return a dict mapping each class to its posterior.

In [ ]:
def naive_bayes(class_sizes, word_counts, present, smoothing=False):
    # your code here
    raise NotImplementedError

In [ ]:
prep.test(naive_bayes, [
    ((class_sizes, word_counts, {"error"}), {"alert": 8/9, "normal": 1/9}),
    ((class_sizes, word_counts, {"error", "ok"}), {"alert": 0.0, "normal": 1.0}),
    ((class_sizes, word_counts, {"error", "ok"}, True), {"alert": (1/54) / (1/54 + 0.196/3)}),
    (({"a": 1, "b": 1}, {"a": {"w": 1}, "b": {"w": 0}}, {"w"}, True), {"a": 2/3, "b": 1/3}),
])

### EXPLORE: does smoothing matter with more data?
**PREDICT:** a word appears in 3/4 of a class's training messages. With smoothing, what does P(w | c) become if the class has 4 messages? 40? 400 (with the same 3/4 fraction)?

In [ ]:
prep.table([[n, 0.75 * n, 0.75, (0.75 * n + 1) / (n + 2)] for n in [4, 40, 400]],
           header=["N_c", "count", "raw fraction", "smoothed"])

Smoothing pulls estimates toward 1/2. With little data it pulls hard; with lots of data it barely matters. That's what you want: be cautious only when you know little.

**Now do practice problem 2.3 on paper.** Watch the absent word.

In [ ]:
prep.check("2.3a", ...)   # floor(100 x P(spam | email)), no smoothing
prep.check("2.3b", ...)   # with Laplace smoothing

---
## 7. Elo: from a rating gap to a win probability

**Idea.** A chess rating is one number per player, designed so that the *difference* between two ratings predicts who wins. The **Elo formula**:

**P(R beats S) = 1 ÷ (1 + 10^((S − R)/400))**

Read it piece by piece:
- Only the gap R − S matters.
- Equal ratings: 10⁰ = 1, so P = 1/2.
- In odds (section 5): P ÷ (1 − P) = 10^((R − S)/400). **Every 400 points of advantage multiplies the odds by 10.**
- Fractional powers of 10 work like this: 10^0.5 = √10, because 10^0.5 × 10^0.5 = 10¹. And 10^(−x) = 1 ÷ 10^x.

You won't have a calculator, so here are the powers you'll need (rating gaps in steps of 100):

| x | 0.25 | 0.5 | 0.75 | 1 | 1.25 |
|---|---|---|---|---|---|
| 10^x | 1.778 | 3.162 | 5.623 | 10 | 17.78 |

**Smallest example.** Gaps that are multiples of 400 need no table at all.

In [ ]:
rows = []
for gap in [0, 400, 800, -400]:
    odds = 10 ** (gap / 400)
    rows.append([gap, odds, f"{odds:g}/(1+{odds:g})", odds / (1 + odds)])
prep.table(rows, header=["gap R - S", "odds = 10^(gap/400)", "P = odds/(1+odds)", "P(R beats S)"])

**PREDICT:** the gap doubles from 400 to 800. Did the win probability double? What happened to the odds? Check the table.

### BUILD 8: `elo_p`

In [ ]:
def elo_p(r, s):
    # your code here
    raise NotImplementedError

In [ ]:
prep.test(elo_p, [
    ((1500, 1500), 0.5),
    ((1900, 1500), 10/11),
    ((1500, 1900), 1/11),
    ((2300, 1500), 100/101),
])

### EXPLORE: the shape of the curve
**PREDICT:** sketch P(R beats S) against the gap from −800 to +800. Is it a straight line? Where is it steepest? Then check that P(R beats S) + P(S beats R) = 1 for a few random pairs. Why *must* that be true when there are no draws?

In [ ]:
if prep.ready(elo_p, 1500, 1500):
    gaps = np.linspace(-800, 800, 161)
    plt.figure(figsize=(6, 3.5))
    plt.plot(gaps, [elo_p(g, 0) for g in gaps])
    plt.xlabel("rating gap R - S"); plt.ylabel("P(R beats S)"); plt.title("Elo is an S-curve"); plt.grid(alpha=.3)
    plt.show()
    rng = np.random.default_rng(1)
    for r, s in rng.integers(1200, 2400, size=(3, 2)):
        print(f"R={r}, S={s}:  P(R wins) + P(S wins) = {elo_p(r, s) + elo_p(s, r):.6f}")

This S-shaped curve is the **logistic** (or sigmoid) curve. Logistic regression uses the same shape to turn a score into a probability. It's steepest when the players are even: 100 points matters a lot between equals and hardly at all between a master and a beginner.

**Now do practice problem 2.4(a) on paper.**

In [ ]:
prep.check("2.4a", ...)   # P(1800 beats 1600), to 3 decimal places

---
## 8. A series: enumerate the sequences

**Idea.** Games are independent, so the probability of a whole *sequence* of results is the product of the probabilities of each game (the multiplication rule; independence means each factor doesn't depend on the earlier games). To find P(win the series): **list every sequence, keep the ones the player wins, and add up their probabilities.**

**Smallest example.** Best-of-3, first to 2 wins, and the player wins each game with p = 0.6. For now, pretend all 3 games are always played.

In [ ]:
p = 0.6
rows, total = [], 0
for seq in itertools.product("WL", repeat=3):
    pr = math.prod(p if g == "W" else 1 - p for g in seq)
    wins = seq.count("W")
    rows.append(["".join(seq), " x ".join(f"{p if g == 'W' else 1 - p:.1f}" for g in seq), pr, wins, wins >= 2])
    total += pr if wins >= 2 else 0
prep.table(rows, header=["sequence", "product", "probability", "wins", "series won?"])

**PREDICT:** add up the "series won" rows yourself. Is P(series) bigger or smaller than 0.6?

In [ ]:
print(f"P(at least 2 wins out of 3) = {total:.3f}")

**Wait.** A real best-of-3 stops as soon as someone has 2 wins. "WWL" never happens: the third game isn't played. Is the answer above wrong?

**PREDICT:** list the series as they *really* happen (stop at 2 wins) and add up the winning ones. Same number or different?

In [ ]:
real = ["WW", "WLW", "LWW", "LL", "LWL", "WLL"]
rows, total_real = [], 0
for seq in real:
    pr = math.prod(p if g == "W" else 1 - p for g in seq)
    won = seq.count("W") == 2
    rows.append([seq, pr, won])
    total_real += pr if won else 0
prep.table(rows, header=["real series", "probability", "won?"])
print(f"P(win series) = {total_real:.3f}")

Same answer. Imagine the third game is always played but ignored once the series is decided. "WW" splits into "WWW" and "WWL", and their probabilities add back to p² × (p + (1 − p)) = p². A phantom game never changes who reached 2 wins first. So **"first to 2 wins" = "at least 2 wins out of 3 games"**, and the second is much easier to enumerate.

### BUILD 9: `p_win_at_least`
Return the probability of winning **at least k** of the games, where game i is won with probability `probs[i]`, independently. Enumerate every win/loss sequence (`itertools.product` helps). The games may have *different* probabilities. You'll need that in the chess walkthrough, where each board has its own pairing.

In [ ]:
def p_win_at_least(k, probs):
    # your code here
    raise NotImplementedError

In [ ]:
prep.test(p_win_at_least, [
    ((1, [0.5]), 0.5),
    ((2, [0.5, 0.5, 0.5]), 0.5),
    ((2, [0.6, 0.6, 0.6]), 0.648),
    ((2, [1.0, 0.5, 0.0]), 0.5),
    ((3, [0.5, 0.5, 0.5]), 0.125),
    ((0, [0.3]), 1.0),
])

### EXPLORE: how a longer series amplifies an edge
**PREDICT:** a player wins each game with p = 0.6. How long a series (best of n) is needed before they win the *series* at least 75% of the time? And what happens to a player with p = 0.52?

In [ ]:
if prep.ready(p_win_at_least, 1, [0.5]):
    ns = list(range(1, 16, 2))
    plt.figure(figsize=(6, 3.5))
    for pp in [0.52, 0.55, 0.6, 0.7]:
        ys = [p_win_at_least((n + 1) // 2, [pp] * n) for n in ns]
        plt.plot(ns, ys, marker="o", label=f"p = {pp}")
    plt.xticks(ns); plt.xlabel("best of n games"); plt.ylabel("P(favourite wins the series)")
    plt.title("Longer series favour the favourite"); plt.legend(); plt.grid(alpha=.3)
    plt.show()

**Now do practice problem 2.4(b) and (c) on paper.** For (c), let the plot above guide your sentence.

In [ ]:
prep.check("2.4b", ...)   # floor(100 x P(1800 player wins best-of-3))

**2.4(c), in one sentence:** *Your sentence here.*

---
## 9. Simpson's paradox: when the total disagrees with every part

**Idea.** The success rate of a whole group is a **weighted average** of its subgroups' rates, weighted by how many cases fall in each subgroup. If two things you're comparing have very different subgroup mixes, the overall comparison can point the *opposite* way to every subgroup comparison. That reversal is called **Simpson's paradox**.

**Smallest example.** You test two grippers, suction and fingers, on easy objects (boxes) and awkward objects (cables, soft bags).

In [ ]:
grip = {"suction": {"easy": (8, 10), "awkward": (30, 90)},
        "fingers": {"easy": (63, 90), "awkward": (2, 10)}}
rows = []
for g, d in grip.items():
    for kind, (ok, n) in d.items():
        rows.append([g, kind, f"{ok}/{n}", ok / n])
prep.table(rows, header=["gripper", "objects", "successes", "rate"])

**PREDICT:** suction wins on easy objects *and* on awkward objects. Which gripper has the higher **overall** success rate?

In [ ]:
for g, d in grip.items():
    ok = sum(v[0] for v in d.values()); n = sum(v[1] for v in d.values())
    parts = " + ".join(f"({v[1]}/{n}) x {v[0] / v[1]:.3f}" for v in d.values())
    print(f"{g:8s} overall = {ok}/{n} = {ok / n:.2f}   = weighted average: {parts}")

Look at *who got which objects*. Suction was tested mostly on awkward objects; fingers mostly on easy ones. Object difficulty affects both **which gripper was used** and **whether the grasp succeeded**. A hidden variable that influences both the "treatment" and the "outcome" is called a **confounder**. When there's a confounder, compare within its groups, or design the test so that both grippers get the same mix of objects.

**Association is not causation.** Data you merely *observed* (rather than an experiment where you randomly assigned grippers to objects) can show that two things go together. It can't by itself *prove* that one causes the other, because a confounder you didn't measure might explain the pattern.

### EXPLORE: watch the aggregate reverse
Keep every within-group rate fixed (suction 80% easy / 33% awkward; fingers 70% / 20%) and keep fingers' mix at 90% easy. Now slide the share of easy objects that *suction* is tested on.

**PREDICT:** at what share of easy objects does suction's overall rate overtake fingers'?

In [ ]:
share = np.linspace(0, 1, 101)
suction_overall = share * 0.8 + (1 - share) * (30 / 90)
fingers_overall = 0.9 * 0.7 + 0.1 * 0.2
plt.figure(figsize=(6, 3.5))
plt.plot(share, suction_overall, label="suction overall")
plt.axhline(fingers_overall, color="C1", label="fingers overall (90% easy)")
plt.axvline(0.1, color="gray", ls=":", label="the mix in our data")
plt.xlabel("share of suction's tests on easy objects"); plt.ylabel("overall success rate")
plt.title("Same skill, different mix"); plt.legend(); plt.grid(alpha=.3)
plt.show()

Suction's skill never changed along this line. Only its mix did. The overall rate measures the mix as much as the skill.

**Now do practice problem 2.5 on paper.** Compute the four within-school rates and the two overall rates, then read every option literally. Option D is a job for section 3.

In [ ]:
prep.check("2.5", ...)   # letter

---
## 10. A two-state Markov chain

**Idea.** Each minute your robot's Wi-Fi is either **Connected (C)** or **Dropped (D)**. What happens next depends only on the state *now*, not on the history. That's called the **Markov property**. The four numbers P(next | now) are the **transition probabilities**. Write them as a table with **rows = now** and **columns = next**, so each row adds to 1:

| now \ next | C | D |
|---|---|---|
| C | 0.9 | 0.1 |
| D | 0.5 | 0.5 |

A **distribution** [P(C), P(D)] says how likely each state is at one moment. To step it forward one minute, add up every way to end up in each state (the law of total probability from section 4):

P(C next) = P(C now) · 0.9 + P(D now) · 0.5

**Smallest example.** Start connected, [1, 0], and also start dropped, [0, 1]. Step both forward.

In [ ]:
W = [[0.9, 0.1], [0.5, 0.5]]
rows = []
a, b = [1.0, 0.0], [0.0, 1.0]
for minute in range(9):
    rows.append([minute, a[0], b[0]])
    a = [a[0] * W[0][0] + a[1] * W[1][0], a[0] * W[0][1] + a[1] * W[1][1]]
    b = [b[0] * W[0][0] + b[1] * W[1][0], b[0] * W[0][1] + b[1] * W[1][1]]
prep.table(rows, header=["minute", "P(C), started connected", "P(C), started dropped"])

**PREDICT** (before you look closely at the bottom rows): where does P(C) settle? Does the answer depend on where you started?

### The steady state, by balancing flows
Counting again: imagine 1,000 robots. Say 800 are connected and 200 are dropped. In the next minute, 10% of the 800 drop (80 robots), and 50% of the 200 reconnect (100 robots). So the connected count *rises* by 20. It stops changing only when the two flows are equal:

(number connected) × 0.1 = (number dropped) × 0.5

**PREDICT:** solve that on paper, together with "connected + dropped = 1,000". Compare with the table.

<details><summary>Check</summary>

Connected = 5 × dropped, so dropped = 1000/6 ≈ 167 and connected ≈ 833. The fraction is 5/6 ≈ 0.833, matching the table.

</details>

**The aha: in the long run, the flow out of each state equals the flow into it.** A distribution that one step leaves unchanged is called the **steady state** (or stationary distribution). It's a **fixed point** of the step: put it in, get the same thing out.

### BUILD 10: `markov_step`
`dist` is a list of probabilities, one per state. `P[i][j]` is P(next = j | now = i), so rows are "now" and columns are "next". Return the distribution one step later. Make it work for any number of states.

In [ ]:
def markov_step(dist, P):
    # your code here
    raise NotImplementedError

In [ ]:
prep.test(markov_step, [
    (([1, 0], W), [0.9, 0.1]),
    (([0.5, 0.5], W), [0.7, 0.3]),
    (([0, 1], [[0, 1], [1, 0]]), [1, 0]),
    (([1, 0, 0], [[0, 1, 0], [0, 0, 1], [1, 0, 0]]), [0, 1, 0]),
])

### BUILD 11: `steady_state_2`
A two-state chain: `a` = P(state 0 → state 1) and `b` = P(state 1 → state 0). Return `[π0, π1]`, the long-run fractions, by balancing flows: π0 · a = π1 · b, with π0 + π1 = 1.

In [ ]:
def steady_state_2(a, b):
    # your code here
    raise NotImplementedError

In [ ]:
prep.test(steady_state_2, [
    ((0.1, 0.5), [5/6, 1/6]),
    ((0.5, 0.5), [0.5, 0.5]),
    ((0.3, 0.1), [0.25, 0.75]),
    ((1, 1), [0.5, 0.5]),
])

### EXPLORE: does every chain settle?
**PREDICT:** the "flip-flop" chain always switches state: P = [[0, 1], [1, 0]]. Start it in state 0. Does the distribution settle? What fraction of the time does it spend in each state in the long run?

In [ ]:
if prep.ready(markov_step, [1, 0], W) and prep.ready(steady_state_2, 0.1, 0.5):
    plt.figure(figsize=(6, 3.5))
    for name, P_, start in [("Wi-Fi, start C", W, [1, 0]), ("Wi-Fi, start D", W, [0, 1]),
                            ("flip-flop, start 0", [[0, 1], [1, 0]], [1, 0])]:
        d, ys = start, []
        for _ in range(15):
            ys.append(d[0]); d = markov_step(d, P_)
        plt.plot(ys, marker=".", label=name)
    plt.axhline(steady_state_2(0.1, 0.5)[0], color="gray", ls="--", label="Wi-Fi steady state")
    plt.xlabel("step"); plt.ylabel("P(state 0)"); plt.legend(); plt.grid(alpha=.3)
    plt.title("Settling, and not settling")
    plt.show()
    print("flip-flop, flows balanced:", steady_state_2(1, 1))

The Wi-Fi chain forgets where it started. The flip-flop chain never settles, but balancing flows still gives the right **long-run fraction of time** (half in each state). That long-run fraction is what "steady state" questions on the exam ask for.

**Now do practice problem 2.6 on paper.** For (a), either step twice, or list the two-step paths that start and end in W.

In [ ]:
prep.check("2.6a", ...)   # floor(100 x P(working two hours from now))
prep.check("2.6b", ...)   # floor(100 x long-run fraction working)

---
## 11. Exam walkthrough: Jan 18, "The Pit-Stop Paradox"

> Do this **after** taking the Jan 18 paper as your diagnostic. Open `papers/2026-stage1-jan18/paper.pdf`, page 9.

You get two tables: P(strategy | weather), and P(Hamilton wins | strategy, weather). The prior chance of rain is 50%. You learn only that **H won**. You're asked for P(at least one driver pitted | H wins).

### Step 1: what is hidden, and what is observed?
Write down the hidden things, how many combinations they make, and the observed thing.

<details><summary>Hint</summary>

Two hidden things: the weather (2 options) and the strategy pair (4 options). So there are 2 × 4 = 8 hidden scenarios. The observed thing is "H won".

</details>

*Your answer here.*

### Step 2: the prior of each scenario
Each scenario is a branch: first the weather, then the strategy given the weather. Multiply along the branch (section 2). Here are the paper's numbers:

In [ ]:
strategies = ["neither", "only V", "only H", "both"]
p_rain = 0.5
p_strategy = {"no rain": [0.45, 0.30, 0.10, 0.15], "rain": [0.05, 0.10, 0.10, 0.75]}
p_h_wins   = {"no rain": [0.25, 0.80, 0.10, 0.55], "rain": [0.50, 0.05, 0.95, 0.60]}

In [ ]:
# Your attempt: print the 8-row table: weather, strategy, prior P(weather and strategy),
# P(H wins | scenario), and joint P(weather and strategy and H wins).

<details><summary>Hint</summary>

P(rain and both pit) = P(rain) × P(both pit | rain). The eight priors must add to 1. Then each joint is prior × P(H wins | scenario): the joint table of section 4, with 8 rows instead of 2.

</details>

### Step 3: check your table
Copy your 8 priors and 8 joints below, in the order "no rain" rows first, then "rain", each in the order neither / only V / only H / both. The checker compares them with the paper's data and tells you which rows are off. It never prints the answer.

In [ ]:
my_prior = [..., ..., ..., ..., ..., ..., ..., ...]
my_joint = [..., ..., ..., ..., ..., ..., ..., ...]

def check_pitstop(prior, joint):
    if any(x is ... for x in prior + joint):
        print("Fill in all 16 numbers first."); return
    k = 0
    for weather, pw in [("no rain", 1 - p_rain), ("rain", p_rain)]:
        for i, s in enumerate(strategies):
            ok_p = math.isclose(prior[k], pw * p_strategy[weather][i], abs_tol=1e-4)
            ok_j = math.isclose(joint[k], pw * p_strategy[weather][i] * p_h_wins[weather][i], abs_tol=1e-4)
            print(f"{weather:8s} {s:8s}  prior {'ok' if ok_p else 'CHECK'}   joint {'ok' if ok_j else 'CHECK'}")
            k += 1
    print(f"your priors add to {sum(prior):.4f} (must be 1)")

check_pitstop(my_prior, my_joint)

**A side question (planned confusion).** The paper says each driver "independently chooses" whether to pit. Test that claim on the no-rain table with your section-3 function. (The cell below builds the 2×2 table, scaled to counts per 100 races: rows = V pits / V doesn't, columns = H pits / H doesn't.) What do you find? Does it matter for this problem?

In [ ]:
if prep.ready(is_independent, [[1, 1], [1, 1]]):
    t = [[15, 30],   # V pits:        both, only V
         [10, 45]]   # V doesn't pit: only H, neither
    print("no-rain table independent?", is_independent(t))

<details><summary>What to take from this</summary>

Independence would need P(both) = P(V pits) × P(H pits). The table doesn't satisfy that, so the word and the table disagree. Luckily you never need independence here: the table already gives the probability of each strategy *pair*, so you never multiply single-driver probabilities. Compute with the numbers you're given, not with assumptions from the story (move M1 from notebook 01).

</details>

### Step 4: zoom into "at least one pitted"
P(H wins) is the sum of all 8 joints (the law of total probability). Which of the 8 rows belong to "at least one driver pitted"?

<details><summary>Hint 1</summary>

Every strategy except "neither" has at least one driver pitting: 6 of the 8 rows.

</details>

<details><summary>Hint 2</summary>

Two routes. Add those 6 joints and divide by P(H wins). Or use the complement from section 1: 1 − (the 2 "neither" joints) ÷ P(H wins). The complement is less arithmetic. Do both if you have time; they must agree.

</details>

In [ ]:
prep.check("J18-pitstop", ...)   # floor(100 x P(at least one pitted | H wins))

---
## 12. Exam walkthrough: Jan 18, "The Blind Board Gambit"

> Do this **after** taking the Jan 18 paper as your diagnostic. Open `papers/2026-stage1-jan18/paper.pdf`, page 7.

Each team puts its 3 players on 3 boards. A team wins the match by winning at least 2 of the 3 boards. Board results are independent and follow the Elo formula. Gukesh: A = 2000, B = 1800, C = 1600. Anand: X = 1900, Y = 1700, Z = 1500.

### Step 1: read the setup as counting
How many ways can each team assign its players to the boards? Which of Gukesh's assignments are still possible after the spy's tip?

<details><summary>Hint</summary>

An ordering of 3 players is a **permutation**: 3 choices for board 1, then 2 for board 2, then 1, so 3 × 2 × 1 = 6. The tip removes (A, B, C), which leaves 5.

</details>

In [ ]:
gukesh = {"A": 2000, "B": 1800, "C": 1600}
anand = {"X": 1900, "Y": 1700, "Z": 1500}
gukesh_orders = [o for o in itertools.permutations("ABC") if o != ("A", "B", "C")]
anand_orders = list(itertools.permutations("XYZ"))
print("Gukesh's possible orders (after the tip):", ["".join(o) for o in gukesh_orders])
print("Anand's possible orders:", ["".join(o) for o in anand_orders])

### Step 2: one match is section 8
Fix one Anand order and one Gukesh order. Which of your BUILD functions gives P(Anand wins the match)?

<details><summary>Hint</summary>

`p_win_at_least(2, [p1, p2, p3])`, where p_i = `elo_p(rating of Anand's board-i player, rating of Gukesh's board-i player)`.

</details>

Only **9 different pairings** exist (3 Anand players × 3 Gukesh players), and every match uses three of them: one from each row and one from each column. So on paper, make the 3×3 table of P(Anand's player wins) **first**, using the powers-of-10 table from section 7. Then check it here (3 decimals is plenty):

In [ ]:
# rows: X, Y, Z (Anand) ; columns: A, B, C (Gukesh) ; entry = P(Anand's player wins that board)
my_pairings = [
    [..., ..., ...],
    [..., ..., ...],
    [..., ..., ...],
]

def check_pairings(T):
    if any(x is ... for row in T for x in row):
        print("Fill in all nine cells first."); return
    for i, a in enumerate("XYZ"):
        marks = []
        for j, g in enumerate("ABC"):
            true = 1 / (1 + 10 ** ((gukesh[g] - anand[a]) / 400))
            marks.append(f"{a} vs {g}: {'ok' if abs(T[i][j] - true) < 0.002 else 'CHECK'}")
        print("   ".join(marks))

check_pairings(my_pairings)

### Step 3: what "maximises expected win probability" means
Anand doesn't know Gukesh's order. He believes each of the 5 remaining orders has probability 1/5. The **expected value** of a quantity is its probability-weighted average. So for each of his 6 orders, Anand averages P(Anand wins the match) over the 5 Gukesh orders, and picks the order with the biggest average.

The brute-force job is a 6 × 5 table: rows = Anand's orders, columns = Gukesh's orders, entries = match-win probabilities, then one average per row. Try it in code (your BUILD functions make it short).

In [ ]:
# Your attempt: for each Anand order, the average of P(Anand wins the match) over the 5 Gukesh orders.

On paper, 30 matches is far too many. There's a shortcut.

<details><summary>Hint 1</summary>

Board numbers don't matter. Only *who plays whom* matters.

</details>

<details><summary>Hint 2</summary>

Fix one Anand order and let Gukesh run through **all 6** orders, including (A, B, C). How many of the 6 possible pairings does Anand meet? What does that say about the average over all 6 orders, for each Anand order? (Run the cell below to test your claim.)

</details>

In [ ]:
if prep.ready(elo_p, 1500, 1500) and prep.ready(p_win_at_least, 1, [0.5]):
    for a in anand_orders:
        avg6 = sum(p_win_at_least(2, [elo_p(anand[a[i]], gukesh[g[i]]) for i in range(3)])
                   for g in itertools.permutations("ABC")) / 6
        print("Anand order", "".join(a), " average over ALL 6 Gukesh orders:", round(avg6, 4))

<details><summary>Hint 3</summary>

Call that common average K. The average over the 5 allowed orders is (6K − P(win against A, B, C)) ÷ 5. To make it as big as possible, Anand wants the order that does **worst** against (A, B, C). Now you only need 6 match probabilities, not 30.

</details>

### Step 4: the match that actually happens
Gukesh actually plays (C, A, B). Anand chose his order by *averaging over his beliefs*, but the result is **not** an average: p is P(Anand wins the match) for his chosen order against (C, A, B) only.

In [ ]:
prep.check("J18-chess", ...)   # floor(100 p)

---
## 13. Connect

Answer in your own words in the cell below. Then copy the best sentence into `log/ERROR_LOG.md` under "Moves I own".

1. In notebook 01, section 3, you computed the gripper detector's **precision** by imagining 1,000 frames. In section 4 here, you computed **P(obstacle | beep)**. Show that these are the same calculation. In each one, which group did you zoom into: a truth group or a prediction group?
2. The Markov steady state is a **fixed point**: one more step leaves it unchanged. In Set 04 you'll meet PageRank, which is the same idea with many web pages instead of two states. What would "flows balance" mean for a single web page?
3. Simpson's paradox happened because an overall rate is a **weighted average** of group rates. In notebook 01, accuracy is a weighted average of two rates. Which two, and what are the weights? Why did that make "95% accuracy" misleading for the gripper?

*Your answers here.*